# 10v1 - A2 v2: pooled 4-fold CV (spec section 5.2 gate)

**Follows `09v1`** (A2 v2 fold-0 pilot: gold macro 0.7956 vs. A2 v1's
0.7689 baseline, macro check passed, but the directional read on
`medial_meniscus_tear`/`lateral_meniscus_tear` was **inconclusive** -
`medial_meniscus_tear` moved +0.1392 (real) but `lateral_meniscus_tear`
moved -0.0005 (flat), so the paired "both move positive" rule didn't
trigger). Per spec section 5.1's own explicit rule, scaling to the
remaining 3 folds anyway is a judgement call for the user, not
automatic - the user made that call, so this notebook implements spec
section 5.2's real statistical gate on the pooled 4-fold OOF result.

Full design/gate reference:
`docs/superpowers/specs/2026-08-28-a2v2-multigroup-slot-attention-design.md`
section 5.2 (approved after two Opus review passes). Self-contained per
this project's Kaggle constraint (no `import src`), same hand-kept-copy
pattern as `05v2`/`06v2`/`09v1`.

**Design, matching `06v2`'s own precedent for A2 v1:**
- **Fold 0**: reuse `09v1`'s real trained checkpoint
  (`a2_v2_fold0_best.pt`) for **inference only** - no retraining. Upload
  `models/a2_v2_fold0_best.pt` (gitignored, downloaded from the `09v1`
  Kaggle run) as a new Kaggle Dataset and attach it to this kernel, then
  edit `CHECKPOINT_PATHS[0]` below if the path differs. Reuse is only
  trusted after it reproduces `09v1`'s real recorded score
  (0.7956, asserted below) - a mismatch means the fold split or checkpoint
  path is wrong, not that reuse itself is invalid.
- **Folds 1, 2, 3**: train fresh, same architecture/hyperparameters as
  `09v1`'s fold-0 run (18 pseudo-slots, `MICRO_BATCH=8`/
  `ACCUMULATE_STEPS=4` - the real values `09v1` measured fit a T4's
  16GB VRAM, ~6.73GB peak; the pre-flight cell below re-checks this
  hasn't regressed, same purpose as `06v2`'s own pre-flight). Real cost:
  ~2-2.5h/fold at 18 pseudo-slots (`09v1`'s own estimate, ~3x A2 v1's
  6-slot compute) - up to ~6-7.5h total for the 3 folds, within the
  30h/week Kaggle quota but close to a single session's practical
  length. **Resume mechanism, same as `06v2` needed for A2 v1's fold 3:**
  if this session is interrupted partway through, and a fold's **all 12
  epochs finished** (its `fold N: best gold macro-AUC=...` summary line
  was printed - an interrupted, mid-training checkpoint must NOT be
  reused, it would put an under-trained model into the pooled comparison
  and bias the section 5.2 gate), download that checkpoint, re-upload as
  a Dataset, and add an entry to `CHECKPOINT_PATHS` (with its matching
  `EXPECTED_REUSED_GOLD_AUC` entry, read from that fold's final `best
  gold macro-AUC=` line, not an intermediate epoch's) before re-running -
  `TRAIN_FOLDS` below is derived from `CHECKPOINT_PATHS`'s keys, so an
  added entry is skipped on retraining automatically.

**The actual gate, spec section 5.2 (not fold-0's section 5.1 macro-only
check):**
1. `per_label_gate(baseline, candidate, tol=0.03, min_concordant=3)` on
   the original 4-finding weak cluster (`mcl_injury`,
   `oa_lateral_compartment`, `medial_meniscus_tear`,
   `lateral_meniscus_tear`), pooled baselines 0.6145/0.6325/0.6635/0.6584
   (A2 v1's own real pooled result, `06v2`). Decision, fixing the sign
   bug the first spec review caught: hypothesis supported only if
   `broad_effect` is true **AND** `macro_delta > 0` - `per_label_gate`
   alone reports a broad *move*, not a broad *improvement*.
2. Gold macro regression check: pooled 12-finding macro vs. A2 v1's
   pooled baseline 0.7512, `gold_tol=0.03`.

**Graduation decision:** both checks pass -> graduate
`expand_slot_groups()` to `src/features.py` and `SlotCacheDataset`'s
`expand_groups=True` path to `src/dataset.py`. Either check fails -> do
not graduate; report the real result.

In [ ]:
import hashlib
import re
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

_KAGGLE_RAW = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
ON_KAGGLE = _KAGGLE_RAW.exists()
if not ON_KAGGLE:
    raise RuntimeError(
        "This notebook needs the full DICOM tree + GPU + the A3 cache "
        "attached - run on Kaggle, not locally."
    )
RAW_DIR = _KAGGLE_RAW
CACHE_DIR = Path("/kaggle/input/datasets/alherma7/cache-stevenleehans-rsna/cache")

# Not part of the official competition mount - a separately-attached
# Kaggle Dataset. EDIT this path to match wherever it actually lands
# under /kaggle/input/ once attached (check `!ls /kaggle/input` if unsure).
PUBLISHED_LABELS_PATH = Path("/kaggle/input/llm-labels-v4-blend/llm_labels_v4_blend.csv")

# Fold 0's checkpoint from 09v1's real run - upload
# models/a2_v2_fold0_best.pt as a Kaggle Dataset and attach it, then EDIT
# this path if it differs. Add fold 1/2/3 entries here only if resuming
# a partially-completed run (see this notebook's own intro).
CHECKPOINT_PATHS = {
    0: Path("/kaggle/input/a2-v2-fold0-checkpoint/a2_v2_fold0_best.pt"),
}
# Each reused checkpoint's real recorded gold macro-AUC from its own
# training run - used below to confirm reuse is valid (fold split /
# checkpoint path both correct), not just assumed.
EXPECTED_REUSED_GOLD_AUC = {
    0: 0.7956,  # 09v1's real fold-0 result, 11-finding mean (oa_lateral_compartment undefined that fold)
}
assert set(CHECKPOINT_PATHS) == set(EXPECTED_REUSED_GOLD_AUC), (
    "CHECKPOINT_PATHS and EXPECTED_REUSED_GOLD_AUC must have matching fold-id keys - "
    "adding a resume entry to only one of the two dicts fails with a bare KeyError below "
    "instead of this readable message"
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
print("CACHE_DIR:", CACHE_DIR, "exists:", CACHE_DIR.exists())
print("PUBLISHED_LABELS_PATH:", PUBLISHED_LABELS_PATH, "exists:", PUBLISHED_LABELS_PATH.exists())
for fold_id, path in CHECKPOINT_PATHS.items():
    print(f"CHECKPOINT_PATHS[{fold_id}]:", path, "exists:", path.exists())

FINDINGS = [
    "acl_injury", "mcl_injury", "medial_meniscus_tear", "lateral_meniscus_tear",
    "oa_medial_compartment", "oa_lateral_compartment", "oa_patellofemoral_compartment",
    "effusion", "synovitis", "bakers_cyst", "bone_contusion", "fracture",
]
OFFICIAL_LABEL_COLUMNS = {
    "acl_injury": "ACL", "mcl_injury": "MCL",
    "medial_meniscus_tear": "Medial Meniscus", "lateral_meniscus_tear": "Lateral Meniscus",
    "oa_medial_compartment": "Medial OA", "oa_lateral_compartment": "Lateral OA",
    "oa_patellofemoral_compartment": "PF OA", "effusion": "Effusion",
    "synovitis": "Synovitis", "bakers_cyst": "Baker's",
    "bone_contusion": "Contusion", "fracture": "Fracture",
}
SLOT_NAMES = ["SAG_FLUID_FS", "COR_FLUID_FS", "AX_FLUID_FS", "SAG_FLUID_NOFS", "COR_T1", "SAG_T1"]
SLOT_CACHE_GROUP_SIZE = 3
SLOT_CACHE_N_GROUPS = 3
N_SLOTS = len(SLOT_NAMES) * SLOT_CACHE_N_GROUPS  # 18 pseudo-slots, same as 09v1
CV_FOLDS = 4
TRAIN_SHARDS = [f"train.s{i:02d}of04" for i in range(4)]

BATCH_SIZE = 32  # effective batch, matches 09v1/A2 v1 - isolates group_index/pseudo-slot count as the only changed variable
MICRO_BATCH = 8  # 09v1's real measured value: 6.73GB peak of 15.6GB (T4) at micro-batch 8
ACCUMULATE_STEPS = BATCH_SIZE // MICRO_BATCH
assert BATCH_SIZE % MICRO_BATCH == 0, "MICRO_BATCH must evenly divide BATCH_SIZE"
EPOCHS = 12
print(f"BATCH_SIZE={BATCH_SIZE}, MICRO_BATCH={MICRO_BATCH}, ACCUMULATE_STEPS={ACCUMULATE_STEPS}")

## Labels: gold official values + A1a' published set for weak studies

In [ ]:
def load_published_labels(path):
    published = pd.read_csv(path)
    label_cols = list(OFFICIAL_LABEL_COLUMNS.values())
    published = published.set_index("StudyInstanceUID")[label_cols]
    published.columns = list(OFFICIAL_LABEL_COLUMNS.keys())
    return published


def load_gold_labels(raw_dir):
    train = pd.read_csv(raw_dir / "train.csv")
    label_cols = list(OFFICIAL_LABEL_COLUMNS.values())
    gold_mask = train[label_cols].notna().all(axis=1)
    gold = train.loc[gold_mask, ["StudyInstanceUID"] + label_cols].set_index("StudyInstanceUID")
    gold.columns = list(OFFICIAL_LABEL_COLUMNS.keys())
    return gold


train_csv = pd.read_csv(RAW_DIR / "train.csv")
reports = train_csv.set_index("StudyInstanceUID")[["Report"]]
gold = load_gold_labels(RAW_DIR)
published = load_published_labels(PUBLISHED_LABELS_PATH)

missing = set(train_csv["StudyInstanceUID"]) - set(published.index)
print("train.csv studies missing from published labels:", len(missing))
assert len(missing) == 0

is_gold = reports.index.isin(gold.index)
label_table = published.reindex(reports.index)[FINDINGS].copy()
label_table.loc[gold.index, FINDINGS] = gold[FINDINGS]
label_table["is_gold"] = is_gold
print(label_table.shape, "gold rows:", label_table["is_gold"].sum())
assert label_table["is_gold"].sum() == 58

## Folds: report-template + scanner-fingerprint grouping (A0)

Identical logic to `05v2`/`06v2`/`09v1`'s fold-assignment cell -
`GroupKFold` has no shuffling/randomness, so recomputing from the same
inputs reproduces the exact same split. Asserted against `09v1`'s real
recorded fold-0 val set (1,307 studies, 17 gold) before trusting the
reused fold-0 checkpoint's predictions.

In [ ]:
def report_group_key(report_text):
    if not isinstance(report_text, str):
        normalized = ""
    else:
        t = unicodedata.normalize("NFKD", report_text.lower())
        t = "".join(ch for ch in t if not unicodedata.combining(ch))
        normalized = re.sub(r"\s+", " ", t).strip()
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


SCANNER_FINGERPRINT_TAGS = (
    "Manufacturer", "ManufacturerModelName", "InstitutionName",
    "DeviceSerialNumber", "MagneticFieldStrength", "StationName",
)


def build_scanner_fingerprints(raw_dir, split="train"):
    series = pd.read_csv(raw_dir / f"{split}_series.csv")
    first_series = series.drop_duplicates("StudyInstanceUID", keep="first")
    fingerprints = {}
    for row in first_series.itertuples(index=False):
        series_dir = raw_dir / f"{split}_series" / row.StudyInstanceUID / row.SeriesInstanceUID
        files = sorted(series_dir.glob("*.dcm"))
        if not files:
            fingerprints[row.StudyInstanceUID] = None
            continue
        ds = pydicom.dcmread(files[0], stop_before_pixels=True)
        fingerprints[row.StudyInstanceUID] = tuple(
            str(getattr(ds, tag, None)) for tag in SCANNER_FINGERPRINT_TAGS
        )
    result = pd.Series(fingerprints, name="scanner_fingerprint")
    result.index.name = "StudyInstanceUID"
    return result


def build_group_ids(*group_key_series):
    index = group_key_series[0].index
    parent = {i: i for i in index}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    for keys in group_key_series:
        valid = keys.dropna()
        for _, idx in valid.groupby(valid).groups.items():
            idx = list(idx)
            for other in idx[1:]:
                union(idx[0], other)

    return pd.Series({i: find(i) for i in index}, name="group_id")


t0 = time.time()
scanner_fp = build_scanner_fingerprints(RAW_DIR, split="train")
print(f"scanner fingerprints: {time.time() - t0:.1f}s for {len(scanner_fp)} studies")

group_keys = reports["Report"].apply(report_group_key)
group_ids = build_group_ids(group_keys, scanner_fp.reindex(reports.index))

gkf = GroupKFold(n_splits=CV_FOLDS)
fold = pd.Series(-1, index=reports.index, dtype=int)
for fold_idx, (_, val_idx) in enumerate(gkf.split(reports, groups=group_ids.to_numpy())):
    fold.iloc[val_idx] = fold_idx
label_table["fold"] = fold
print(label_table["fold"].value_counts().sort_index())

fold0_val = label_table[label_table["fold"] == 0]
print(f"fold 0: {len(fold0_val)} val ({int(fold0_val['is_gold'].sum())} gold)")
assert len(fold0_val) == 1307, f"expected 1307 val studies in fold 0, got {len(fold0_val)}"
assert int(fold0_val["is_gold"].sum()) == 17, f"expected 17 gold in fold 0, got {int(fold0_val['is_gold'].sum())}"
print("fold assignment matches 09v1's real run - safe to reuse the fold-0 checkpoint")

## Reshape: `select_group()` (A3, unchanged) + `expand_slot_groups()` (A2 v2, unchanged from `09v1`)

In [ ]:
def select_group(cache_slot_stack, group_index):
    if isinstance(group_index, int):
        group_index = [group_index]
    size = SLOT_CACHE_GROUP_SIZE
    groups = [cache_slot_stack[..., g * size:(g + 1) * size, :, :] for g in group_index]
    return np.concatenate(groups, axis=-3)


def expand_slot_groups(cache_slot_stack, slot_mask):
    '''cache_slot_stack: (n_slots, 9, H, W) - one study, all slots.
    slot_mask: (n_slots,). Returns (images, mask):
    images (n_slots * SLOT_CACHE_N_GROUPS, SLOT_CACHE_GROUP_SIZE, H, W),
    slot-major order (pseudo-slot index = s * SLOT_CACHE_N_GROUPS + g).
    mask (n_slots * SLOT_CACHE_N_GROUPS,), each real slot's bit repeated
    SLOT_CACHE_N_GROUPS times. Per approved spec section 2.1.'''
    n_slots = cache_slot_stack.shape[0]
    h, w = cache_slot_stack.shape[-2:]
    groups = [select_group(cache_slot_stack, g) for g in range(SLOT_CACHE_N_GROUPS)]
    stacked = np.stack(groups, axis=1)  # (n_slots, n_groups, 3, H, W)
    images = stacked.reshape(n_slots * SLOT_CACHE_N_GROUPS, SLOT_CACHE_GROUP_SIZE, h, w)
    mask = np.repeat(slot_mask, SLOT_CACHE_N_GROUPS)
    return images, mask


# Self-validate: each channel's pixels are set to that channel's global
# index, so the expected output is exact and checkable by hand.
_demo_stack = np.zeros((6, 9, 4, 4), dtype=np.uint8)
for _c in range(9):
    _demo_stack[:, _c] = _c
_demo_mask = np.array([1.0, 0.0, 1.0, 1.0, 0.0, 1.0], dtype=np.float32)
_images, _mask = expand_slot_groups(_demo_stack, _demo_mask)
assert _images.shape == (18, 3, 4, 4)
assert _mask.shape == (18,)
for _s in range(6):
    for _g in range(3):
        assert np.array_equal(_images[_s * 3 + _g], select_group(_demo_stack[_s], _g))
assert np.array_equal(_mask, np.repeat(_demo_mask, 3))
print("expand_slot_groups matches select_group per (slot, group) pair and replicates the mask - OK")

## Cache dataset

Identical to `09v1` - opens all 4 train shards as memmaps, `expand_groups=True`
selects all 3 anchor groups on all 6 slots. `full_ds` is built once here
(unlike `09v1`, which built it inline in the fold-0 training cell) since
it's reused across the fold-0 reuse pass and all 3 fresh-training folds
below, same convention as `06v2`.

In [ ]:
class SlotCacheDataset(torch.utils.data.Dataset):
    def __init__(self, cache_dir, shards, labels_df, group_index=1, expand_groups=False, study_ids=None):
        if expand_groups and group_index != 1:
            raise ValueError(
                "expand_groups=True ignores group_index; pass group_index=1 (default) or omit "
                f"it, not group_index={group_index!r}"
            )
        self.group_index = group_index
        self.expand_groups = expand_groups
        caches, masks, all_study_ids, shard_of, local_idx = [], [], [], [], []
        for shard in shards:
            cache = np.load(cache_dir / f"{shard}_cache.npy", mmap_mode="r")
            mask = np.load(cache_dir / f"{shard}_mask.npy")
            studies = pd.read_csv(cache_dir / f"{shard}_studies.csv")
            caches.append(cache)
            masks.append(mask)
            all_study_ids.append(studies["StudyInstanceUID"].to_numpy())
            shard_of.append(np.full(len(studies), len(caches) - 1))
            local_idx.append(np.arange(len(studies)))

        self.caches = caches
        mask_all = np.concatenate(masks, axis=0).astype(np.float32)
        study_ids_all = np.concatenate(all_study_ids)
        shard_of_all = np.concatenate(shard_of)
        local_idx_all = np.concatenate(local_idx)

        if study_ids is not None:
            keep = np.isin(study_ids_all, np.asarray(list(study_ids)))
            mask_all, study_ids_all = mask_all[keep], study_ids_all[keep]
            shard_of_all, local_idx_all = shard_of_all[keep], local_idx_all[keep]

        self.mask = mask_all
        self.study_ids = study_ids_all
        self.shard_of = shard_of_all
        self.local_idx = local_idx_all

        aligned = labels_df.reindex(self.study_ids)[FINDINGS]
        if aligned.isna().any().any():
            missing = self.study_ids[aligned.isna().any(axis=1).to_numpy()]
            raise ValueError(f"{len(missing)} cache studies missing labels, e.g. {missing[:5]}")
        self.labels = aligned.to_numpy(dtype=np.float32)

    def __len__(self):
        return len(self.study_ids)

    def __getitem__(self, i):
        shard_idx, row = self.shard_of[i], self.local_idx[i]
        full = self.caches[shard_idx][row]  # (6, 9, 224, 224) uint8
        if self.expand_groups:
            selected, slot_mask_row = expand_slot_groups(full, self.mask[i])
        else:
            g = self.group_index
            selected = full[:, g * SLOT_CACHE_GROUP_SIZE:(g + 1) * SLOT_CACHE_GROUP_SIZE]
            slot_mask_row = self.mask[i]
        images = torch.from_numpy(np.ascontiguousarray(selected)).float() / 255.0
        mask = torch.from_numpy(slot_mask_row)
        label = torch.from_numpy(self.labels[i])
        return images, mask, label


sanity_ds = SlotCacheDataset(CACHE_DIR, TRAIN_SHARDS[:1], label_table, expand_groups=True)
images, mask, label = sanity_ds[0]
print("images:", images.shape, images.dtype, "mask:", mask.shape, "label:", label.shape)
assert images.shape == (N_SLOTS, 3, 224, 224)
assert mask.shape == (N_SLOTS,)
assert not torch.isnan(images).any()

_raw_mask = sanity_ds.mask[0]
assert np.array_equal(mask.numpy(), np.repeat(_raw_mask, 3)), "real-data mask replication mismatch"
_s = int(np.flatnonzero(_raw_mask)[0])
assert not torch.equal(images[_s * 3], images[_s * 3 + 1]), (
    "groups 0 and 1 are identical on a real present slot - this run would be a no-op vs. A2 v1"
)
print("SlotCacheDataset(expand_groups=True) sanity check + real-data mask/group-distinctness check OK")

try:
    SlotCacheDataset(CACHE_DIR, TRAIN_SHARDS[:1], label_table, expand_groups=True, group_index=0)
    raise AssertionError("expand_groups=True with a non-default group_index should have raised")
except ValueError:
    print("expand_groups=True with group_index=0 correctly raises - OK")

full_ds = SlotCacheDataset(CACHE_DIR, TRAIN_SHARDS, label_table, expand_groups=True)
print(f"full_ds: {len(full_ds)} studies")
# The whole pooled gate below rests on full_ds's row order matching
# label_table's (train_idx/val_idx are positional indices into
# label_table, used to Subset(full_ds, ...)) - asserted explicitly here,
# not just inherited untested from 05v2/06v2/09v1's same assumption.
assert len(full_ds) == len(label_table)
assert np.array_equal(full_ds.study_ids, label_table.index.to_numpy())
print("full_ds row order matches label_table - OK")

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "timm"], check=True)
import timm
print("timm:", timm.__version__)

## Model: DINOv2-small backbone + masked_finding_attention

Unchanged from `09v1` - `n_slots` defaults to `N_SLOTS` (18).

In [ ]:
def masked_finding_attention(embeddings, mask, query, head_weight, head_bias):
    if not (mask.sum(dim=1) > 0).all():
        raise ValueError("masked_finding_attention: a row has 0 present slots")
    scores = torch.einsum("od,bsd->bos", query, embeddings) / (embeddings.shape[-1] ** 0.5)
    expanded_mask = mask.unsqueeze(1).expand(-1, query.shape[0], -1)
    scores = scores.masked_fill(expanded_mask == 0, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    context = torch.einsum("bos,bsd->bod", weights, embeddings)
    logits = (context * head_weight.unsqueeze(0)).sum(-1) + head_bias
    if torch.isnan(logits).any() or torch.isinf(logits).any():
        raise RuntimeError("masked_finding_attention produced NaN/Inf logits")
    return logits


class SlotAttentionModel(nn.Module):
    def __init__(self, n_findings=len(FINDINGS), n_slots=N_SLOTS,
                 backbone_name="vit_small_patch14_dinov2.lvd142m", unfreeze_last=6):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=True, num_classes=0, img_size=224,
        )
        embed_dim = self.backbone.num_features
        for p in self.backbone.parameters():
            p.requires_grad = False
        for block in self.backbone.blocks[-unfreeze_last:]:
            for p in block.parameters():
                p.requires_grad = True

        self.query = nn.Parameter(torch.randn(n_findings, embed_dim) * (embed_dim ** -0.5))
        self.heads = nn.Linear(embed_dim, n_findings)
        self.embed_dim = embed_dim
        self.n_findings = n_findings
        self.n_slots = n_slots

    def forward(self, slot_images, slot_mask):
        B, S, C, H, W = slot_images.shape
        if (S, C, H, W) != (self.n_slots, 3, 224, 224):
            raise ValueError(
                f"expected slot_images (*, {self.n_slots}, 3, 224, 224), got {tuple(slot_images.shape)}"
            )
        if tuple(slot_mask.shape) != (B, S):
            raise ValueError(f"expected slot_mask ({B}, {S}), got {tuple(slot_mask.shape)}")

        flat = slot_images.view(B * S, C, H, W)
        embeddings = self.backbone(flat).view(B, S, self.embed_dim)
        return masked_finding_attention(
            embeddings, slot_mask, self.query, self.heads.weight, self.heads.bias
        )


print("SlotAttentionModel defined (n_slots=18) - instantiating to confirm it loads real DINOv2 weights...")
_smoke_model = SlotAttentionModel()
n_trainable = sum(p.numel() for p in _smoke_model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in _smoke_model.parameters())
print(f"embed_dim={_smoke_model.embed_dim}, n_slots={_smoke_model.n_slots}, "
      f"trainable params={n_trainable:,} / {n_total:,}")
del _smoke_model

## Evaluation helpers

Hand-kept copies of `src/evaluate.py::macro_roc_auc`/`per_finding_roc_auc`/`per_label_gate`
(the last one newly needed here for the spec section 5.2 gate - not
used at fold-0 tier in `09v1`).

In [ ]:
def per_finding_roc_auc(y_true, y_pred):
    scores = {}
    for c in y_true.columns:
        if y_true[c].nunique() < 2:
            scores[c] = float("nan")
        else:
            scores[c] = roc_auc_score(y_true[c], y_pred[c])
    return pd.Series(scores)


def macro_roc_auc(y_true, y_pred):
    per_finding = per_finding_roc_auc(y_true, y_pred)
    undefined = per_finding[per_finding.isna()]
    if len(undefined) > 0:
        print(f"  (macro_roc_auc: {len(undefined)} finding(s) undefined this fold "
              f"- {list(undefined.index)}, excluded from the mean, not treated as 0)")
    return float(per_finding.mean())


def per_label_gate(baseline_auc, candidate_auc, tol=0.03, min_concordant=7):
    '''Hand-kept copy of src/evaluate.py::per_label_gate. min_concordant=3
    is passed explicitly below (a bare majority of the 4-finding weak
    cluster), not this function's own default of 7 (sized for the full
    12-finding set).'''
    delta = candidate_auc - baseline_auc
    macro_delta = float(delta.mean())
    moved = delta[delta.abs() >= tol]
    concordant = int((np.sign(moved) == np.sign(macro_delta)).sum()) if macro_delta != 0 else 0
    return {
        "macro_delta": macro_delta,
        "n_labels_moved": int(len(moved)),
        "n_concordant": concordant,
        "broad_effect": concordant >= min_concordant,
        "per_label_delta": delta,
    }

## Pre-flight: overfit 8 real studies

Same architecture/wiring as `09v1`, already validated there (real
fold-0 run completed cleanly); this just confirms nothing regressed
(timm/torch version drift, this session's GPU type) before spending
~6-7.5h on folds 1-3's real training runs, same purpose as `06v2`'s own
pre-flight before its fold-3 run.

In [ ]:
model = SlotAttentionModel().to(DEVICE)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                         lr=1e-3, weight_decay=0.02)

tiny_studies = label_table.index[:8]
tiny_ds = SlotCacheDataset(CACHE_DIR, TRAIN_SHARDS, label_table, expand_groups=True, study_ids=tiny_studies)
# batch_size=MICRO_BATCH (not a hard-coded 8) so peak_gb below is directly
# comparable to real training - if MICRO_BATCH is ever edited from its
# default of 8, this pre-flight measures the actual value that will run.
tiny_loader = torch.utils.data.DataLoader(tiny_ds, batch_size=MICRO_BATCH, shuffle=False)
images, mask, labels = next(iter(tiny_loader))
images, mask, labels = images.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
bytes_per_batch = images.element_size() * images.nelement()
print(f"images tensor: {tuple(images.shape)}, {bytes_per_batch / 1e6:.1f} MB for this "
      f"batch of {images.shape[0]} studies at 18 pseudo-slots (~3x A2 v1's 6-slot size) "
      "- host-RAM sanity check per spec section 4.")

eps = 1e-7
loss_floor = -(labels * torch.log(labels.clamp(eps, 1)) +
               (1 - labels) * torch.log((1 - labels).clamp(eps, 1))).mean().item()
print(f"loss floor for this batch (soft-label entropy): {loss_floor:.4f}")

print("pre-flight: overfitting 8 real studies...")
model.train()
for step in range(500):
    opt.zero_grad()
    loss = F.binary_cross_entropy_with_logits(model(images, mask), labels)
    loss.backward()
    opt.step()
    if step % 100 == 0:
        print(f"  step {step}: loss={loss.item():.4f}")
print(f"final pre-flight loss: {loss.item():.4f} (floor: {loss_floor:.4f})")
assert loss.item() < loss_floor + 0.02, (
    f"model failed to reach the soft-label loss floor ({loss_floor:.4f}) on 8 real "
    f"studies - stop and debug before the real runs"
)
if DEVICE.type == "cuda":
    peak_gb = torch.cuda.max_memory_allocated() / 1e9
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"peak VRAM during pre-flight (micro-batch={MICRO_BATCH}): {peak_gb:.2f} GB of {total_gb:.1f} GB total")
    # Compares directly against MICRO_BATCH's own measured peak - gradient
    # accumulation means BATCH_SIZE (32) is never resident at once, so
    # scaling by BATCH_SIZE/MICRO_BATCH here would ask the wrong question.
    if peak_gb >= 0.85 * total_gb:
        print("WARNING: this session's GPU has less headroom than 09v1's - consider lowering "
              "MICRO_BATCH (and raising ACCUMULATE_STEPS to match, keeping their product at "
              "BATCH_SIZE) before the real training cells below.")
print("pre-flight OK")
del model, opt

## Training function

Refactors `09v1`'s single-fold training loop (gradient accumulation,
scheduler stepped on optimizer steps only) into a reusable function, so
folds 1-3 can each call it without copy-pasting the loop 3 times.
Checkpoints to `/kaggle/working/a2_v2_fold{fold_id}_best.pt` and returns
that fold's val predictions (all studies, gold and weak) plus the val
label table, for pooling later - same return shape as `06v2`'s
`train_fold`.

In [ ]:
def train_fold(fold_id, epochs=EPOCHS):
    # Reads MICRO_BATCH/ACCUMULATE_STEPS from the module-level globals at
    # call time (not as default args, which would bind stale values if
    # cell 1 is edited+re-run without also re-running this cell) - same
    # convention eval_checkpoint_on_fold already uses for MICRO_BATCH.
    micro_batch, accumulate_steps = MICRO_BATCH, ACCUMULATE_STEPS
    train_idx = np.flatnonzero(label_table["fold"].to_numpy() != fold_id)
    val_idx = np.flatnonzero(label_table["fold"].to_numpy() == fold_id)
    val_labels = label_table.iloc[val_idx].reset_index()
    val_is_gold = val_labels["is_gold"].to_numpy()
    print(f"fold {fold_id}: {len(train_idx)} train / {len(val_idx)} val ({val_is_gold.sum()} gold in val)")

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(full_ds, train_idx.tolist()), batch_size=micro_batch,
        shuffle=True, num_workers=2, drop_last=True,
        # drop_last=True avoids a partial accumulation cycle at each
        # epoch's end, which would otherwise need special-casing below.
    )
    val_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(full_ds, val_idx.tolist()), batch_size=micro_batch, shuffle=False, num_workers=2,
    )

    model = SlotAttentionModel().to(DEVICE)
    backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("backbone")]
    head_params = [p for n, p in model.named_parameters() if not n.startswith("backbone")]
    opt = torch.optim.AdamW([
        {"params": backbone_params, "lr": 8e-6},
        {"params": head_params, "lr": 1e-3},
    ], weight_decay=0.02)

    # total_steps counts *optimizer* steps, not micro-batches - with
    # accumulate_steps>1 an optimizer step only happens every
    # accumulate_steps micro-batches, so the scheduler must be built (and
    # stepped below) on that basis, matching 09v1's fold-0 cell exactly.
    steps_per_epoch = len(train_loader) // accumulate_steps
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[8e-6, 1e-3], total_steps=epochs * steps_per_epoch
    )

    ckpt_path = f"/kaggle/working/a2_v2_fold{fold_id}_best.pt"
    best_gold_auc = -1.0
    best_val_pred = None
    for epoch in range(epochs):
        model.train()
        t0 = time.time()
        opt.zero_grad()
        for step, (images, mask, labels) in enumerate(train_loader):
            images, mask, labels = images.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
            loss = F.binary_cross_entropy_with_logits(model(images, mask), labels) / accumulate_steps
            loss.backward()
            if (step + 1) % accumulate_steps == 0:
                opt.step()
                opt.zero_grad()
                scheduler.step()

        model.eval()
        probs = []
        with torch.no_grad():
            for images, mask, _ in val_loader:
                images, mask = images.to(DEVICE), mask.to(DEVICE)
                probs.append(torch.sigmoid(model(images, mask)).cpu().numpy())
        val_pred = pd.DataFrame(np.concatenate(probs), columns=FINDINGS)
        gold_auc = macro_roc_auc(val_labels.loc[val_is_gold, FINDINGS], val_pred[val_is_gold])
        print(f"  epoch {epoch}: {time.time() - t0:.0f}s, val gold macro-AUC={gold_auc:.4f}")

        if gold_auc > best_gold_auc:
            best_gold_auc = gold_auc
            best_val_pred = val_pred.copy()
            torch.save(model.state_dict(), ckpt_path)
            print("    new best, checkpoint saved")

    print(f"fold {fold_id}: best gold macro-AUC={best_gold_auc:.4f}, checkpoint saved to {ckpt_path}")
    del model, opt
    return val_labels, best_val_pred, best_gold_auc


def eval_checkpoint_on_fold(fold_id, checkpoint_path):
    '''Scores an existing checkpoint on its own fold's val set - inference
    only, no training. Valid as an OOF read because the checkpoint was
    trained with this fold held out.'''
    val_idx = np.flatnonzero(label_table["fold"].to_numpy() == fold_id)
    val_labels = label_table.iloc[val_idx].reset_index()
    val_is_gold = val_labels["is_gold"].to_numpy()
    print(f"fold {fold_id}: {len(val_idx)} val ({val_is_gold.sum()} gold in val), reusing checkpoint")

    val_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(full_ds, val_idx.tolist()), batch_size=MICRO_BATCH, shuffle=False, num_workers=2,
    )

    model = SlotAttentionModel().to(DEVICE)
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    model.eval()
    probs = []
    with torch.no_grad():
        for images, mask, _ in val_loader:
            images, mask = images.to(DEVICE), mask.to(DEVICE)
            probs.append(torch.sigmoid(model(images, mask)).cpu().numpy())
    val_pred = pd.DataFrame(np.concatenate(probs), columns=FINDINGS)
    del model

    gold_auc = macro_roc_auc(val_labels.loc[val_is_gold, FINDINGS], val_pred[val_is_gold])
    print(f"fold {fold_id} (reused checkpoint): gold macro-AUC={gold_auc:.4f}")
    return val_labels, val_pred, gold_auc

## Fold 0: reuse existing checkpoint (inference only)

No training - loads `09v1`'s real fold-0 checkpoint and scores it once
on fold 0's own val set. Checked against its recorded training-run
score (0.7956) before trusting it, since a mismatch would mean
`CHECKPOINT_PATHS[0]` points at the wrong file or the fold split doesn't
match the run that trained it.

In [ ]:
REUSE_FOLDS = sorted(CHECKPOINT_PATHS.keys())
TRAIN_FOLDS = [f for f in range(CV_FOLDS) if f not in REUSE_FOLDS]
print("REUSE_FOLDS (inference only):", REUSE_FOLDS)
print("TRAIN_FOLDS (train fresh):", TRAIN_FOLDS)

fold_results = {}
for fold_id in REUSE_FOLDS:
    expected = EXPECTED_REUSED_GOLD_AUC[fold_id]
    val_labels, val_pred, gold_auc = eval_checkpoint_on_fold(fold_id, CHECKPOINT_PATHS[fold_id])
    assert abs(gold_auc - expected) < 0.01, (
        f"fold {fold_id}'s reused checkpoint scored {gold_auc:.4f}, expected ~{expected:.4f} - "
        f"check CHECKPOINT_PATHS[{fold_id}] points at the right file and the fold split matches "
        f"the run that trained it"
    )
    fold_results[fold_id] = (val_labels, val_pred, gold_auc)

print("reused checkpoint(s) match their recorded training-run score(s) - reuse is valid")

## Folds 1, 2, 3: train fresh

Same architecture/hyperparameters as `09v1`'s fold-0 run, ~2-2.5h/fold.

In [ ]:
for fold_id in TRAIN_FOLDS:
    fold_results[fold_id] = train_fold(fold_id)

## Pooled OOF report: all 4 folds, all 58 gold studies, spec section 5.2 gate

Concatenates every fold's gold predictions (each study appears in
exactly one fold's val set) into one 58-row table, then runs the two
real spec section 5.2 checks - `per_label_gate` on the 4-finding weak
cluster and the pooled gold macro regression check - for the actual
graduation decision.

In [ ]:
gold_true_parts, gold_pred_parts = [], []
for fold_id in sorted(fold_results):
    val_labels, val_pred, _ = fold_results[fold_id]
    is_gold = val_labels["is_gold"].to_numpy()
    gold_true_parts.append(val_labels.loc[is_gold, FINDINGS].reset_index(drop=True))
    gold_pred_parts.append(val_pred[is_gold].reset_index(drop=True))

pooled_true = pd.concat(gold_true_parts, ignore_index=True)
pooled_pred = pd.concat(gold_pred_parts, ignore_index=True)
print(f"pooled gold studies: {len(pooled_true)} (expected 58)")
assert len(pooled_true) == 58

pooled_per_finding = per_finding_roc_auc(pooled_true, pooled_pred)
print("\npooled per-finding AUC (all 58 gold, 4-fold OOF):\n", pooled_per_finding)
assert pooled_per_finding.notna().all(), (
    "a finding is undefined (single-class) even pooled across all 58 gold studies - "
    "the macro/weak-cluster checks below would silently drop it from their mean/gate "
    "instead of comparing against a real 12-finding / 4-finding baseline"
)
pooled_macro_12 = float(pooled_per_finding.mean())
print(f"\npooled gold macro-AUC (12-finding, 4-fold OOF): {pooled_macro_12:.4f}")

print("\nfold-by-fold gold macro-AUC (for reference, noisier - 11-17 gold each):")
for fold_id in sorted(fold_results):
    print(f"  fold {fold_id}: {fold_results[fold_id][2]:.4f}")

# ---- spec section 5.2, check 1: per_label_gate on the 4-finding weak cluster ----
WEAK_CLUSTER = ["mcl_injury", "oa_lateral_compartment", "medial_meniscus_tear", "lateral_meniscus_tear"]
BASELINE_CLUSTER_AUC = pd.Series({
    "mcl_injury": 0.6145,
    "oa_lateral_compartment": 0.6325,
    "medial_meniscus_tear": 0.6635,
    "lateral_meniscus_tear": 0.6584,
})  # A2 v1's real pooled result, 06v2
candidate_cluster_auc = pooled_per_finding[WEAK_CLUSTER]
print("\ncandidate cluster AUC:\n", candidate_cluster_auc)

label_check = per_label_gate(BASELINE_CLUSTER_AUC, candidate_cluster_auc, tol=0.03, min_concordant=3)
print("\nper_label_gate (weak cluster, tol=0.03, min_concordant=3):")
for k, v in label_check.items():
    print(f"  {k}: {v}")

# Fixing the sign bug the first spec review caught: broad_effect alone
# reports a broad MOVE, not a broad IMPROVEMENT - require macro_delta>0 too.
hypothesis_supported = label_check["broad_effect"] and label_check["macro_delta"] > 0
print(f"\nhypothesis supported (broad_effect AND macro_delta>0): {hypothesis_supported}")

# ---- spec section 5.2, check 2: pooled gold macro regression check ----
BASELINE_POOLED_MACRO_12 = 0.7512  # A2 v1's real pooled result, 06v2
GOLD_TOL = 0.03
macro_regression_delta = pooled_macro_12 - BASELINE_POOLED_MACRO_12
macro_regression_ok = macro_regression_delta >= -GOLD_TOL
print(f"\npooled gold macro (12-finding): {pooled_macro_12:.4f} vs. A2 v1 pooled baseline "
      f"{BASELINE_POOLED_MACRO_12}: delta={macro_regression_delta:+.4f}, "
      f"ok={macro_regression_ok} (tol={GOLD_TOL})")

print()
if hypothesis_supported and macro_regression_ok:
    print("DECISION: both checks pass -> GRADUATE expand_slot_groups() to src/features.py and "
          "SlotCacheDataset's expand_groups=True path to src/dataset.py, per spec section 5.2.")
else:
    print("DECISION: at least one check failed -> DO NOT GRADUATE. Report the real result; "
          "treat hypothesis 2 (all 3 anchor groups beat centre-only) as not empirically "
          "supported at this scale (spec section 5.2).")

print("\nfull pooled per-finding AUC for the record:\n", pooled_per_finding)

## Real output

Run on Kaggle 2026-08-29 (fold 0 reused for inference, folds 1-3
trained fresh - real total ~25,800s ~= 7.2h). One real gotcha:
`CHECKPOINT_PATHS[0]` needed editing to the real Kaggle Model mount path
(`/kaggle/input/models/alherma7/a2-v2-fold0-best/pytorch/default/1/a2_v2_fold0_best.pt`
- a Model, not a Dataset upload) - expected, same "upload it yourself,
edit the path" convention as every prior checkpoint reuse in this
project.

**Fold assignment**: matched exactly (1,307/1,034/1,033/1,033 val,
17/11/19/11 gold) - fold 0's assertion against `09v1`'s recorded split
passed.

**Pre-flight**: loss 0.2993 (floor 0.2993), peak VRAM 6.73 GB of 15.6 GB
- no VRAM warning this run (confirms the pre-run review's I1 fix:
the old formula scaled by `BATCH_SIZE/MICRO_BATCH` and fired a spurious
warning at this exact same real measurement).

**Fold 0 (reused checkpoint)**: gold macro-AUC=0.7956 - matches its
recorded training-run score exactly, reuse assertion passed.

**Folds 1-3 (trained fresh), best gold macro-AUC per fold:**
```
fold 1: best gold macro-AUC=0.6771 (epoch 6)
fold 2: best gold macro-AUC=0.8693 (epoch 6)
fold 3: best gold macro-AUC=0.8206 (epoch 4)
```

**Pooled OOF report (all 58 gold studies, 4-fold OOF):**

```
pooled gold studies: 58 (expected 58)

pooled per-finding AUC (all 58 gold, 4-fold OOF):
 acl_injury                       0.808824
mcl_injury                       0.659864
medial_meniscus_tear             0.700721
lateral_meniscus_tear            0.690683
oa_medial_compartment            0.917829
oa_lateral_compartment           0.796905
oa_patellofemoral_compartment    0.797941
effusion                         0.915528
synovitis                        0.787336
bakers_cyst                      0.840580
bone_contusion                   0.800270
fracture                         0.894444

pooled gold macro-AUC (12-finding, 4-fold OOF): 0.8009

fold-by-fold gold macro-AUC (for reference, noisier - 11-17 gold each):
  fold 0: 0.7956
  fold 1: 0.6771
  fold 2: 0.8693
  fold 3: 0.8206
```

**Spec section 5.2 gate, both checks computed for real:**

```
candidate cluster AUC:
 mcl_injury                0.659864
oa_lateral_compartment    0.796905
medial_meniscus_tear      0.700721
lateral_meniscus_tear     0.690683

per_label_gate (weak cluster, tol=0.03, min_concordant=3):
  macro_delta: 0.0698183879187968
  n_labels_moved: 4
  n_concordant: 4
  broad_effect: True
  per_label_delta: mcl_injury                0.045364
oa_lateral_compartment    0.164405
medial_meniscus_tear      0.037221
lateral_meniscus_tear     0.032283

hypothesis supported (broad_effect AND macro_delta>0): True

pooled gold macro (12-finding): 0.8009 vs. A2 v1 pooled baseline 0.7512: delta=+0.0497, ok=True (tol=0.03)

DECISION: both checks pass -> GRADUATE expand_slot_groups() to src/features.py and
SlotCacheDataset's expand_groups=True path to src/dataset.py, per spec section 5.2.
```

**Both checks passed cleanly, not marginally.** All 4/4 weak-cluster
findings moved concordantly positive (`n_concordant=4` against
`min_concordant=3`) - `oa_lateral_compartment` (the finding excluded
from every earlier-tier read for lacking a fold-0 baseline at all) moved
the most of any of the 4 (+0.1644), and `lateral_meniscus_tear` (whose
flat fold-0 read drove `09v1`'s "inconclusive" verdict) came back
clearly positive once pooled across all 4 folds (+0.0323) - consistent
with this project's own repeated finding that single-fold 17-gold-study
reads are noisy and pooling resolves them, not evidence the fold-0
inconclusive call was wrong. The pooled macro regression check is also
a real improvement (+0.0497), not just a pass.

**Real evidence the multi-group hypothesis (all 3 A3 anchor groups beat
centre-only) holds up** - `expand_slot_groups()`/`SlotCacheDataset`'s
`expand_groups=True` path are the real next candidates for graduation to
`src/`, per spec section 5.2's own graduation rule, pending the user's
explicit go-ahead (same discipline as A2 v1/A3's own graduations).